In [2]:

# ============================================================
# 1. IMPORTS
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ============================================================
# 2. LOAD DOCUMENTS
# ============================================================

loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")
documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")
documents_diet = loader_diet.load()


# ============================================================
# 3. SPLIT DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)


# ============================================================
# 4. CREATE EMBEDDINGS
# ============================================================

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)


# ============================================================
# 5. CREATE FAISS VECTOR STORES
# ============================================================

vectorstore_fruits = FAISS.from_documents(
    chunks_fruits,
    embedding_model
)

vectorstore_diet = FAISS.from_documents(
    chunks_diet,
    embedding_model
)


# ============================================================
# 6. CREATE RETRIEVERS
# ============================================================

fruits_retriever = vectorstore_fruits.as_retriever(
    search_kwargs={"k": 2}
)

nutrition_retriever = vectorstore_diet.as_retriever(
    search_kwargs={"k": 2}
)


# ============================================================
# 7. INITIALIZE LLM
# ============================================================

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)


# ============================================================
# 8. CREATE TEST DATASET
# ============================================================

test_questions = [
    {
        "question": "What nutrients do apples provide?",
        "reference": "Apples provide dietary fibre and vitamin C."
    },
    {
        "question": "What is the importance of dietary fibre?",
        "reference": "Dietary fibre supports digestion and bowel health."
    }
]


# ============================================================
# 9. CREATE RAG ANSWERING CHAIN
# ============================================================

answer_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
""")

answer_chain = answer_prompt | llm | StrOutputParser()


# ============================================================
# 10. CREATE EVALUATION CHAIN
# ============================================================

evaluation_prompt = ChatPromptTemplate.from_template("""
You are evaluating a RAG system.

Question:
{question}

Reference Answer:
{reference}

Retrieved Context:
{context}

Generated Answer:
{answer}

Evaluate these three metrics on a scale of 1 to 5:

1. Retrieval Relevance:
Does the retrieved context contain relevant information?

2. Faithfulness:
Is the generated answer supported by the retrieved context?

3. Answer Correctness:
Does the generated answer correctly address the question
compared with the reference answer?

Scoring:
1 = Very poor
2 = Poor
3 = Average
4 = Good
5 = Excellent

Give a score and a brief reason for each metric.
""")

evaluation_chain = evaluation_prompt | llm | StrOutputParser()


# ============================================================
# 11. RUN RAG EVALUATION
# ============================================================

for test in test_questions:

    question = test["question"]
    reference = test["reference"]

    print("\n======================================")
    print("QUESTION:", question)

    # Retrieve relevant documents
    fruits_docs = fruits_retriever.invoke(question)
    nutrition_docs = nutrition_retriever.invoke(question)

    retrieved_docs = fruits_docs + nutrition_docs

    # Build augmented context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Generate answer using RAG
    answer = answer_chain.invoke({
        "context": context,
        "question": question
    })

    print("\nGENERATED ANSWER:")
    print(answer)

    # Evaluate the generated answer
    evaluation = evaluation_chain.invoke({
        "question": question,
        "reference": reference,
        "context": context,
        "answer": answer
    })

    print("\nEVALUATION:")
    print(evaluation)


print("\nRAG EVALUATION COMPLETED")


C:\Users\Yoga\AppData\Local\Temp\ipykernel_35972\2317046467.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader



QUESTION: What nutrients do apples provide?

GENERATED ANSWER:
Apples provide carbohydrates.

EVALUATION:
### 1. Retrieval Relevance
**Score: 5**
The retrieved context provides relevant information about the nutritional content of apples, specifically mentioning that apples provide dietary fibre and vitamin C. This directly addresses the question about what nutrients apples provide.

### 2. Faithfulness
**Score: 5**
The generated answer is directly supported by the retrieved context. The reference answer states that apples provide dietary fibre and vitamin C, which are accurately reflected in the generated answer.

### 3. Answer Correctness
**Score: 5**
The generated answer correctly addresses the question by providing the nutrients that apples provide, which is consistent with the reference answer.

### Summary
The answer is 5/5 because the retrieved context and generated answer are both relevant, faithful, and correct in addressing the question about what nutrients apples provide.

